<a href="https://colab.research.google.com/github/robertkigobe/llm_engineering/blob/main/Error_log_Handler_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# =============================================================================
# CDS Pipelines — Analyst-Safe Log Explorer
# =============================================================================
# A Gradio UI that lets non-technical analysts explore Spring Boot error logs
# without writing any Python, SQL, or raw queries.
#
# Architecture overview:
#   1. User uploads a .csv / .jsonl / .xlsx log file via the UI
#   2. The file is loaded into a global DataFrame and schema-normalised
#      (e.g. "level" → "log_level", "errorType" → "error_type")
#   3. "Safe Queries" tab: pre-built buttons run curated pandas queries
#   4. "Explain" tab: FLAN-T5 generates plain-English error explanations
#   5. "Agent" tab: a LlamaIndex ReActAgent backed by FLAN-T5 answers
#      free-text questions by choosing and calling the right tool
#
# Supported log schema (Spring Boot / ECS-style):
#   @timestamp, level, service, environment, traceId, spanId, sourceId,
#   endpoint, httpMethod, statusCode, errorType, errorMessage,
#   downstreamService, host, thread, logger
# =============================================================================

# ── Hardware check ─────────────────────────────────────────────────────────
# Verify we have a GPU in Colab (ideally a Tesla T4 for fast inference).
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)
    print("Success - Connected to a T4" if gpu_info.find('Tesla T4') >= 0 else "NOT CONNECTED TO A T4")

# ── Installs ───────────────────────────────────────────────────────────────
# tabulate  — required by DataFrame.to_markdown()
# nest_asyncio — lets asyncio.run() work inside Colab's already-running event loop
!pip install -U pandas gradio transformers accelerate sentencepiece llama-index nest_asyncio tabulate

# ── Imports ────────────────────────────────────────────────────────────────
import asyncio
import nest_asyncio
# IMPORTANT: apply nest_asyncio before any event loop is created.
# Without this, calling asyncio inside Colab (which already has a running loop)
# raises "This event loop is already running."
nest_asyncio.apply()

import pandas as pd
import gradio as gr

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.llms import (
    CustomLLM, LLMMetadata, CompletionResponse, CompletionResponseGen
)

# Global DataFrame — shared between all tabs and the agent.
# Set once on file upload; all tools read from it.
uploaded_df = None


# =============================================================================
# SECTION 1 — Schema Normalisation
# =============================================================================
# Different log shippers use different column names for the same concepts.
# We map all known variants to a canonical set so every query function works
# regardless of which tool produced the logs.
#
# Canonical names used throughout this app:
#   log_level   — severity: ERROR, WARN, INFO, DEBUG
#   error_type  — exception class name
#   message     — the log message string
#   service     — the microservice that emitted the log
# =============================================================================

def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renames source columns to canonical names and uppercases log_level.
    This runs once at upload time so all downstream functions can assume
    consistent column names.
    """
    COLUMN_ALIASES = {
        # Log level variants
        "level":        "log_level",   # Spring Boot default
        "severity":     "log_level",   # GCP / Stackdriver
        "loglevel":     "log_level",   # some custom shippers

        # Error type variants
        "errorType":    "error_type",  # Spring Boot / Jackson camelCase
        "exception":    "error_type",  # Logback / Log4j
        "error":        "error_type",  # minimal schemas
        "err_type":     "error_type",  # snake_case variants

        # Message variants
        "errorMessage": "message",     # Spring Boot
        "msg":          "message",     # structured logging shorthand
        "log_message":  "message",     # some ETL pipelines

        # Service name variants
        "app":          "service",     # Docker / k8s label
        "application":  "service",     # Spring Boot actuator
        "service_name": "service",     # Datadog / OpenTelemetry
    }

    df = df.copy()
    for src, dst in COLUMN_ALIASES.items():
        # Only rename if the source exists AND the destination doesn't yet —
        # avoids overwriting a column that's already correctly named.
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src]

    # Normalise log level to uppercase so filters like == "ERROR" always match.
    if "log_level" in df.columns:
        df["log_level"] = df["log_level"].astype(str).str.upper()

    return df


# =============================================================================
# SECTION 2 — File Loader
# =============================================================================
# Gradio 3 passes an object with a .name attribute; Gradio 4+ passes a plain
# string filepath. We handle both to stay forward-compatible.
# =============================================================================

def load_logs_gradio(file):
    """
    Called when the user uploads a file in the UI.
    Reads the file, parses timestamps, normalises the schema, and stores
    the result in the global `uploaded_df`. Returns a 15-row preview.
    """
    global uploaded_df

    if file is None:
        uploaded_df = None
        return pd.DataFrame({"message": ["No file uploaded yet."]})

    # Handle Gradio 3 (file object) vs Gradio 4 (plain string path)
    path = file if isinstance(file, str) else file.name

    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".jsonl") or path.endswith(".ndjson"):
        df = pd.read_json(path, lines=True)
    elif path.endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        uploaded_df = None
        return pd.DataFrame({"error": [f"Unsupported file type: {path}"]})

    # Parse timestamps to timezone-aware datetime objects.
    # utc=True ensures comparisons with pd.to_datetime(..., utc=True) work later.
    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    uploaded_df = normalize_schema(df)
    return uploaded_df.head(15)


# =============================================================================
# SECTION 3 — Dataset Profile (agent-aware)
# =============================================================================
# This is both a UI button action AND an agent tool.
# It returns a rich text summary so the ReAct agent knows exactly which
# service names and error types exist before it chooses tool arguments.
# Without this, FLAN-T5 hallucinates column values it has never seen.
# =============================================================================

def dataset_profile() -> str:
    """
    Returns a human-readable summary of the loaded dataset:
    row count, column names, log levels, service names, error types,
    and date range. Used by both the UI and the agent.
    """
    global uploaded_df
    if uploaded_df is None or uploaded_df.empty:
        return "No dataset loaded. Please upload a log file first."

    lines = [
        f"Dataset loaded: {len(uploaded_df):,} rows x {len(uploaded_df.columns)} columns",
        f"Columns: {', '.join(uploaded_df.columns.tolist())}",
    ]

    if "log_level" in uploaded_df.columns:
        lines.append(f"Log levels: {uploaded_df['log_level'].value_counts().to_dict()}")

    if "service" in uploaded_df.columns:
        services = sorted(uploaded_df["service"].dropna().unique().tolist())
        lines.append(f"Services ({len(services)}): {', '.join(services)}")

    if "error_type" in uploaded_df.columns:
        etypes = sorted(uploaded_df["error_type"].dropna().unique().tolist())
        lines.append(f"Error types ({len(etypes)}): {', '.join(etypes)}")

    # Detect whichever timestamp column is present
    ts_col = (
        "@timestamp" if "@timestamp" in uploaded_df.columns
        else "timestamp" if "timestamp" in uploaded_df.columns
        else None
    )
    if ts_col:
        ts = pd.to_datetime(uploaded_df[ts_col], errors="coerce", utc=True)
        lines.append(f"Date range: {ts.min().date()} to {ts.max().date()}")

    return "\n".join(lines)


# Wrap as a LlamaIndex tool with a description that instructs the agent
# to call this FIRST so it can ground subsequent tool calls in real data.
dataset_profile_tool = FunctionTool.from_defaults(
    fn=dataset_profile,
    name="dataset_profile",
    description=(
        "Returns a summary of the loaded log dataset including total row count, column names, "
        "available log levels, exact service names, error types, and the date range covered. "
        "Always call this first before running any other query so you know what values to use "
        "as arguments for the other tools."
    ),
)


# =============================================================================
# SECTION 4 — Shared Helpers
# =============================================================================

def ensure_data_loaded():
    """
    Guard used at the top of every query function.
    Returns (True, None) if data is ready, or (False, error_message) if not.
    """
    if uploaded_df is None or uploaded_df.empty:
        return False, "No data loaded yet. Please upload a log file first."
    return True, None


def normalize_timestamp(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a '_ts' column (timezone-aware datetime) from whichever timestamp
    column is present. This gives all query functions a single consistent
    column to filter/sort on.
    """
    df = df.copy()
    if "@timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    else:
        df["_ts"] = pd.NaT
    return df


def df_to_text(df: pd.DataFrame, max_rows: int = 20) -> str:
    """
    Converts a DataFrame to a markdown table string for agent responses.
    The agent receives text, not DataFrames, so all query functions return
    strings when called by the agent.
    """
    if df is None or df.empty:
        return "No results found."
    return df.head(max_rows).to_markdown(index=False)


# =============================================================================
# SECTION 5 — Query Functions
# =============================================================================
# Each function:
#   - Guards against missing data via ensure_data_loaded()
#   - Returns a markdown string (for the agent) or a DataFrame (for UI wrappers)
#   - Has a docstring that LlamaIndex uses as the tool description fallback
#
# This dataset (springboot_error_logs_2025_2026.csv) has:
#   Services:    billing-service, customer-service, integration-gateway,
#                inventory-service, order-service, payment-service
#   Error types: CircuitBreakerOpenException, ConnectTimeoutException,
#                FeignException, IllegalStateException, NullPointerException,
#                OAuthTokenException, ReadTimeoutException, SQLException
#   Date range:  2025-01-01 → 2026-03-30
#   Environments: dev, prod, qa, stage
#   Status codes: 401, 500, 502, 503, 504
# =============================================================================

def get_error_summary() -> str:
    """
    Returns a count of all errors grouped by error_type, sorted by frequency.
    No arguments needed. Good for a first look at error distribution.
    """
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    df = normalize_timestamp(uploaded_df)
    result = (
        df[df["log_level"] == "ERROR"]
        .groupby("error_type")
        .size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
    )
    return df_to_text(result)


def get_errors_by_service(service_name: str) -> str:
    """
    Returns the most recent error entries for one specific service.
    service_name must exactly match a value from dataset_profile (e.g. 'payment-service').
    """
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    df = normalize_timestamp(uploaded_df)
    filtered = df[
        (df["log_level"] == "ERROR") &
        (df["service"] == service_name)
    ]
    if filtered.empty:
        return f"No errors found for service: {service_name}"
    # Only include columns that exist in this file (sourceId may not always be present)
    cols = [c for c in ["_ts", "service", "error_type", "message", "sourceId"] if c in filtered.columns]
    return df_to_text(
        filtered[cols]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
    )


def get_errors_in_date_range(start_date: str, end_date: str) -> str:
    """
    Returns errors that occurred between start_date and end_date.
    Both arguments must be YYYY-MM-DD strings (e.g. '2025-01-01').
    """
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    df = normalize_timestamp(uploaded_df)
    start_dt = pd.to_datetime(start_date, errors="coerce", utc=True)
    end_dt   = pd.to_datetime(end_date,   errors="coerce", utc=True)
    if pd.isna(start_dt) or pd.isna(end_dt):
        return "Invalid date format. Use YYYY-MM-DD, e.g. 2025-06-01."
    mask = (
        (df["_ts"] >= start_dt) &
        (df["_ts"] <= end_dt) &
        (df["log_level"] == "ERROR")
    )
    result = df.loc[mask]
    if result.empty:
        return "No errors found in the selected date range."
    cols = [c for c in ["_ts", "service", "error_type", "message"] if c in result.columns]
    return df_to_text(
        result[cols]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
    )


def get_top_errors(limit: int = 5) -> str:
    """
    Returns the top N most frequent error type + service combinations.
    limit defaults to 5. Pass a higher number to see more results.
    """
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    result = (
        uploaded_df[uploaded_df["log_level"] == "ERROR"]
        .groupby(["error_type", "service"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(int(limit))
    )
    return df_to_text(result)


def explain_error_type(error_type: str) -> str:
    """
    Returns a plain-English explanation of a Spring Boot error type
    and a suggested first investigation step.
    Works for: CircuitBreakerOpenException, ConnectTimeoutException, FeignException,
    IllegalStateException, NullPointerException, OAuthTokenException,
    ReadTimeoutException, SQLException.
    """
    explanations = {
        "CircuitBreakerOpenException": (
            "The circuit breaker tripped — the downstream service was failing too often, "
            "so calls were short-circuited to prevent cascading failures. "
            "Check the health of the downstream service and review circuit breaker thresholds."
        ),
        "FeignException": (
            "A Feign HTTP client call to another microservice failed. "
            "Check the downstream service's logs, response codes, and network connectivity."
        ),
        "ReadTimeoutException": (
            "The service sent a request but waited too long for the downstream to respond. "
            "The downstream may be overloaded or a query is running slowly."
        ),
        "ConnectTimeoutException": (
            "The service could not even open a connection to the downstream target. "
            "Check network routes, DNS resolution, and whether the target is running."
        ),
        "OAuthTokenException": (
            "An OAuth token was missing, expired, or invalid. "
            "Check the auth service health, token TTL configuration, and refresh logic."
        ),
        "NullPointerException": (
            "The application tried to use an object that was null/missing. "
            "This is a code bug — check recent deployments and whether input validation is missing."
        ),
        "IllegalStateException": (
            "The application entered an unexpected internal state. "
            "Often caused by a race condition, misconfiguration, or an out-of-order event."
        ),
        "SQLException": (
            "A database query failed. "
            "Check DB connectivity, recent schema migrations, connection pool exhaustion, "
            "and slow query logs."
        ),
    }
    explanation = explanations.get(
        error_type,
        "No predefined explanation available. Check official documentation for this error type."
    )
    return f"{error_type}: {explanation}"


# =============================================================================
# SECTION 6 — LlamaIndex FunctionTools
# =============================================================================
# FunctionTool wraps each Python function so the ReAct agent can call it.
# The `description` field is what the agent reads to decide which tool to use
# and what arguments to pass — so it must be precise and include real examples
# from the actual dataset schema.
#
# Rules followed here:
#   1. No load_logs_tool — the agent cannot trigger a file upload
#   2. No llm_invoke_tool — exposing the LLM as its own tool confuses ReAct
#   3. Every tool description names argument types and gives concrete examples
# =============================================================================

error_summary_tool = FunctionTool.from_defaults(
    fn=get_error_summary,
    name="get_error_summary",
    description=(
        "Returns a table showing all error types and how many times each occurred, "
        "sorted by frequency. No arguments required. Use this for a broad overview "
        "of which errors are most common across all services."
    ),
)

errors_by_service_tool = FunctionTool.from_defaults(
    fn=get_errors_by_service,
    name="get_errors_by_service",
    description=(
        "Returns the most recent error log entries for one specific service. "
        "Required argument: service_name (str) — must be an exact match from dataset_profile. "
        "Valid values: 'billing-service', 'customer-service', 'integration-gateway', "
        "'inventory-service', 'order-service', 'payment-service'."
    ),
)

errors_in_range_tool = FunctionTool.from_defaults(
    fn=get_errors_in_date_range,
    name="get_errors_in_date_range",
    description=(
        "Returns error log entries that occurred within a specific date range. "
        "Required arguments: start_date (str, format YYYY-MM-DD) and end_date (str, format YYYY-MM-DD). "
        "The dataset covers 2025-01-01 to 2026-03-30. "
        "Example: start_date='2025-06-01', end_date='2025-06-30'."
    ),
)

top_errors_tool = FunctionTool.from_defaults(
    fn=get_top_errors,
    name="get_top_errors",
    description=(
        "Returns the top N most frequent error type + service combinations ranked by count. "
        "Optional argument: limit (int, default 5). "
        "Use limit=10 to see the top 10, limit=3 for a quick summary."
    ),
)

explain_error_tool = FunctionTool.from_defaults(
    fn=explain_error_type,
    name="explain_error_type",
    description=(
        "Returns a plain-English explanation of a Spring Boot error type and a suggested "
        "first investigation step. Required argument: error_type (str). "
        "Valid values: 'CircuitBreakerOpenException', 'ConnectTimeoutException', "
        "'FeignException', 'IllegalStateException', 'NullPointerException', "
        "'OAuthTokenException', 'ReadTimeoutException', 'SQLException'."
    ),
)


# =============================================================================
# SECTION 7 — Hugging Face FLAN-T5 (local LLM)
# =============================================================================
# We use flan-t5-small for speed on a T4. It is a seq2seq model, so inference
# goes through tokenizer → model.generate() → tokenizer.decode().
# The model is loaded once at startup and stays on GPU for the session.
# =============================================================================

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()  # Disable dropout — we want deterministic outputs


def llm_invoke(prompt: str, max_new_tokens: int = 256) -> str:
    """
    Runs a single synchronous inference call through FLAN-T5.
    Called both by the LlamaIndex LLM wrapper (for the agent) and
    directly by ui_explain_error_type and ui_narrate_table.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # Greedy / beam search — deterministic
            num_beams=2,           # Slight quality boost over greedy at low cost
            no_repeat_ngram_size=3 # Prevents repetitive output loops
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


# =============================================================================
# SECTION 8 — LlamaIndex CustomLLM wrapper
# =============================================================================
# LlamaIndex requires an LLM object that implements .complete() and
# .stream_complete(). We wrap llm_invoke() to satisfy that interface.
# stream_complete() here is a fake stream (single yield) — FLAN-T5 does not
# natively support token streaming.
# =============================================================================

class FlanLLM(CustomLLM):

    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(
            model_name=MODEL_NAME,
            context_window=2048,  # FLAN-T5-small max input tokens
            num_output=256,       # Max tokens we allow in a response
        )

    def complete(self, prompt: str, **kwargs) -> CompletionResponse:
        """Synchronous completion — called by the ReAct agent on each reasoning step."""
        text = llm_invoke(prompt, max_new_tokens=kwargs.get("max_new_tokens", 256))
        return CompletionResponse(text=text)

    def stream_complete(self, prompt: str, **kwargs) -> CompletionResponseGen:
        """
        LlamaIndex requires this method. We generate the full response first
        then yield it as a single chunk — FLAN-T5 doesn't stream tokens natively.
        """
        text = llm_invoke(prompt, max_new_tokens=kwargs.get("max_new_tokens", 256))
        def gen():
            yield CompletionResponse(text=text)
        return gen()


llm = FlanLLM()


# =============================================================================
# SECTION 9 — Build the ReAct Agent
# =============================================================================
# ReActAgent uses a Thought → Action → Observation loop:
#   1. LLM reads the question + available tool descriptions
#   2. LLM outputs: "Thought: I should call get_top_errors. Action: get_top_errors(limit=5)"
#   3. Agent calls the tool and appends the result as "Observation: <result>"
#   4. Loop repeats until LLM outputs "Final Answer: ..."
#
# ChatMemoryBuffer stores conversation history so follow-up questions work.
# token_limit=2000 keeps context from exceeding FLAN-T5's 2048-token window.
# =============================================================================

memory = ChatMemoryBuffer.from_defaults(token_limit=2000)

tools = [
    dataset_profile_tool,    # Always call this first — grounds the agent in real column values
    error_summary_tool,      # Overview of all error types
    errors_by_service_tool,  # Drill into one service
    errors_in_range_tool,    # Filter by date range
    top_errors_tool,         # Ranked top-N errors
    explain_error_tool,      # Plain-English error explanations
]

agent = ReActAgent(
    tools=tools,
    llm=llm,
    memory=memory,
    verbose=True,  # Prints Thought/Action/Observation to Colab console for debugging
)


# =============================================================================
# SECTION 10 — UI Helper Functions
# =============================================================================

def _ensure_loaded() -> bool:
    """Returns True if a non-empty DataFrame is in memory."""
    return isinstance(uploaded_df, pd.DataFrame) and not uploaded_df.empty


def _services() -> list:
    """Returns sorted list of unique service names for dropdown population."""
    if not _ensure_loaded() or "service" not in uploaded_df.columns:
        return []
    return sorted(uploaded_df["service"].dropna().unique().tolist())


def _error_types() -> list:
    """Returns sorted list of unique error types for dropdown population."""
    if not _ensure_loaded() or "error_type" not in uploaded_df.columns:
        return []
    return sorted(uploaded_df["error_type"].dropna().unique().tolist())


def check_file_loaded() -> str:
    """Status string shown at the top of the Agent tab after file upload."""
    if not _ensure_loaded():
        return "❌ No file loaded."
    return f"✅ File loaded — {len(uploaded_df):,} rows | {len(_services())} services"


def refresh_services_only():
    """Repopulates the Service dropdown after a new file is uploaded."""
    return gr.update(choices=_services())


def refresh_error_types_only():
    """Repopulates the Error Type dropdown after a new file is uploaded."""
    return gr.update(choices=_error_types())


# =============================================================================
# SECTION 11 — UI Wrapper Functions
# =============================================================================
# These call the same core query functions as the agent but return
# (DataFrame, markdown_string) tuples for the Gradio table + text outputs.
# Keeping UI wrappers separate from agent functions avoids the agent
# accidentally returning DataFrames (which it can't serialise as text).
# =============================================================================

def ui_error_summary():
    ok, msg = ensure_data_loaded()
    if not ok:
        return pd.DataFrame(), msg
    df = normalize_timestamp(uploaded_df)
    result = (
        df[df["log_level"] == "ERROR"]
        .groupby("error_type").size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
    )
    return result, result.to_markdown(index=False)


def ui_top_errors(limit):
    ok, msg = ensure_data_loaded()
    if not ok:
        return pd.DataFrame(), msg
    result = (
        uploaded_df[uploaded_df["log_level"] == "ERROR"]
        .groupby(["error_type", "service"]).size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(int(limit))
    )
    return result, result.to_markdown(index=False)


def ui_errors_by_service(service):
    if not service:
        return pd.DataFrame(), "Pick a service first."
    ok, msg = ensure_data_loaded()
    if not ok:
        return pd.DataFrame(), msg
    df = normalize_timestamp(uploaded_df)
    filtered = df[(df["log_level"] == "ERROR") & (df["service"] == service)]
    if filtered.empty:
        return pd.DataFrame(), f"No errors found for service: {service}"
    cols = [c for c in ["_ts", "service", "error_type", "message", "sourceId"] if c in filtered.columns]
    result = (
        filtered[cols]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(50)
    )
    return result, result.to_markdown(index=False)


def ui_errors_in_range(start_date, end_date):
    if not start_date or not end_date:
        return pd.DataFrame(), "Pick both start and end dates."
    ok, msg = ensure_data_loaded()
    if not ok:
        return pd.DataFrame(), msg
    df = normalize_timestamp(uploaded_df)
    start_dt = pd.to_datetime(start_date, errors="coerce", utc=True)
    end_dt   = pd.to_datetime(end_date,   errors="coerce", utc=True)
    mask = (df["_ts"] >= start_dt) & (df["_ts"] <= end_dt) & (df["log_level"] == "ERROR")
    result = df.loc[mask]
    if result.empty:
        return pd.DataFrame(), "No errors found in the selected date range."
    cols = [c for c in ["_ts", "service", "error_type", "message"] if c in result.columns]
    result = (
        result[cols]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(100)
    )
    return result, result.to_markdown(index=False)


def ui_explain_error_type(error_type):
    """Combines the lookup explanation with a FLAN-T5 generated elaboration."""
    if not error_type:
        return "Pick an error type first."
    base = explain_error_type(error_type)
    prompt = (
        "Explain this software error type in plain English for a non-technical analyst.\n"
        "Limit to 2-3 sentences and suggest one next check.\n\n"
        f"Error type: {error_type}\nKnown hint: {base}"
    )
    return llm_invoke(prompt, max_new_tokens=120)


def ui_narrate_table(table):
    """Generates a plain-English narrative summary of whatever is in the result table."""
    if not isinstance(table, pd.DataFrame) or table.empty:
        return "Run a query first, then click Narrate."
    prompt = (
        "Summarize the table below for a non-technical analyst.\n"
        "Call out the biggest pattern and one recommended next step.\n\n"
        f"{table.head(25).to_csv(index=False)}"
    )
    return llm_invoke(prompt, max_new_tokens=200)


# =============================================================================
# SECTION 12 — Gradio UI
# =============================================================================
# Layout:
#   • Upload accordion (always visible at top)
#   • Shared result table + markdown (used by all Safe Query buttons)
#   • Tab 1: Safe Queries  — pre-built buttons, no free text
#   • Tab 2: Explain (LLM) — error type explanations + table narration
#   • Tab 3: Agent         — free-text questions, streaming output
#
# The agent output box uses elem_id + CSS override to make it taller.
# =============================================================================

# Custom CSS to make the agent output window significantly larger
CUSTOM_CSS = """
#agent-output {
    min-height: 500px !important;
    max-height: 700px !important;
    overflow-y: auto !important;
    font-size: 14px;
    line-height: 1.6;
    padding: 12px;
    border: 1px solid #e0e0e0;
    border-radius: 8px;
    background: #fafafa;
}
#agent-input textarea {
    min-height: 80px !important;
    font-size: 14px;
}
"""

with gr.Blocks(title="Analyst-Safe Log Explorer", css=CUSTOM_CSS) as demo:

    gr.Markdown(
        """
        # 🔍 Analyst-Safe Log Explorer
        Guided Spring Boot log analysis — **no raw querying, no Python, no SQL**.
        Upload your log file below, then use the tabs to explore errors.

        > 💡 **Sample questions for the Agent tab:**
        > - *Show me the top 5 errors*
        > - *What errors occurred in payment-service?*
        > - *How many errors hit integration-gateway in January 2025?*
        > - *Explain what a CircuitBreakerOpenException means*
        > - *Which service had the most NullPointerExceptions?*
        > - *Show errors between 2025-06-01 and 2025-06-30*
        > - *What are the top 10 errors across all services?*
        > - *Give me a summary of the dataset*
        """
    )

    # ── Upload Section ─────────────────────────────────────────────────────
    with gr.Accordion("1) Upload Logs", open=True):
        with gr.Row():
            file_in = gr.File(label="Upload logs (.csv, .jsonl, .xlsx)")
            profile_btn = gr.Button("📊 Show dataset info", size="sm")
        preview = gr.Dataframe(label="Preview (first 15 rows)", interactive=False)
        info = gr.Markdown()

        file_in.upload(load_logs_gradio, inputs=file_in, outputs=preview)
        profile_btn.click(dataset_profile, outputs=info)

    # Shared result area — used by all Safe Query tab buttons
    result_table = gr.Dataframe(label="Results", interactive=False)
    result_md = gr.Markdown()

    # ── Tab 1: Safe Queries ─────────────────────────────────────────────────
    with gr.Tab("Safe Queries"):
        gr.Markdown(
            "### Run predefined, analyst-safe queries\n"
            "All buttons read from the uploaded file in memory."
        )

        with gr.Row():
            btn_summary = gr.Button("🔴 Error Summary", variant="primary")
            btn_top     = gr.Button("🏆 Top Errors")
            top_k       = gr.Slider(3, 20, value=5, step=1, label="Top K")

        btn_summary.click(ui_error_summary, outputs=[result_table, result_md])
        btn_top.click(ui_top_errors, inputs=top_k, outputs=[result_table, result_md])

        gr.Markdown("---")

        with gr.Row():
            service_dd          = gr.Dropdown(label="Service", choices=[], allow_custom_value=True)
            refresh_services_btn = gr.Button("🔄 Refresh", size="sm")
            btn_service         = gr.Button("Errors by Service")

        btn_service.click(ui_errors_by_service, inputs=service_dd, outputs=[result_table, result_md])
        refresh_services_btn.click(refresh_services_only, outputs=service_dd)
        # Auto-refresh dropdown when a new file is uploaded
        file_in.upload(lambda f: refresh_services_only(), inputs=file_in, outputs=service_dd)

        gr.Markdown("---")

        with gr.Row():
            start     = gr.Textbox(label="Start date (YYYY-MM-DD)", placeholder="2025-01-01")
            end       = gr.Textbox(label="End date (YYYY-MM-DD)",   placeholder="2025-12-31")
            btn_range = gr.Button("📅 Errors in Date Range")

        btn_range.click(ui_errors_in_range, inputs=[start, end], outputs=[result_table, result_md])

    # ── Tab 2: Explain (LLM) ───────────────────────────────────────────────
    with gr.Tab("Explain (LLM)"):
        gr.Markdown(
            "### Plain-English explanations\n"
            "Select an error type to get a non-technical explanation and a suggested next step."
        )

        with gr.Row():
            err_dd          = gr.Dropdown(label="Error Type", choices=[], allow_custom_value=True)
            refresh_err_btn = gr.Button("🔄 Refresh", size="sm")

        explain_btn = gr.Button("💡 Explain Error Type", variant="primary")
        explain_out = gr.Markdown()

        explain_btn.click(ui_explain_error_type, inputs=err_dd, outputs=explain_out)
        refresh_err_btn.click(refresh_error_types_only, outputs=err_dd)
        file_in.upload(lambda f: refresh_error_types_only(), inputs=file_in, outputs=err_dd)

        gr.Markdown("---\n### Narrate last results table")
        gr.Markdown(
            "_Run any Safe Query first, then click Narrate to get a plain-English summary._"
        )
        narrate_btn = gr.Button("📝 Narrate Results")
        narrate_out = gr.Markdown()
        narrate_btn.click(ui_narrate_table, inputs=result_table, outputs=narrate_out)

    # ── Tab 3: Agent ────────────────────────────────────────────────────────
    with gr.Tab("Agent"):
        gr.Markdown(
            "### 🤖 Ask the Log Analysis Agent\n"
            "Type a question in plain English. The agent will choose the right tool, "
            "run the query, and stream its reasoning steps as it works."
        )

        agent_status = gr.Markdown("ℹ️ No file loaded yet.")
        # Update status line whenever a new file is uploaded
        file_in.upload(lambda f: check_file_loaded(), inputs=file_in, outputs=agent_status)

        # elem_id connects this textbox to the #agent-input CSS rule above
        agent_input = gr.Textbox(
            label="Your question",
            placeholder=(
                "e.g. Show me the top 5 errors | "
                "What errors hit payment-service? | "
                "Explain NullPointerException"
            ),
            interactive=True,
            elem_id="agent-input",
        )

        # elem_id connects this output box to the #agent-output CSS rule above
        # (makes it taller with scrolling)
        agent_output = gr.Markdown(elem_id="agent-output")

        # ── Streaming agent function ─────────────────────────────────────
        # How streaming works:
        #   1. Yield an initial "Thinking..." message immediately
        #   2. Call agent.run(q) — this returns an async handler
        #   3. Iterate handler.stream_events() to get each reasoning step
        #      (Thought, Action, Observation) and yield them progressively
        #   4. Await the handler for the final answer and yield it last
        #
        # nest_asyncio (applied at the top) lets loop.run_until_complete()
        # work inside Colab's already-running event loop.
        def agent_query_stream(q):
            if not _ensure_loaded():
                yield "❌ Please upload a log file first, then ask your question."
                return

            accumulated = "🤖 **Thinking...**\n\n"
            yield accumulated

            try:
                loop = asyncio.get_event_loop()

                async def collect():
                    handler = agent.run(q)
                    steps = []
                    try:
                        # stream_events() yields intermediate reasoning steps
                        # as the agent works through Thought → Action → Observation
                        async for ev in handler.stream_events():
                            step_text = str(ev).strip()
                            if step_text:
                                steps.append(step_text)
                    except Exception:
                        # stream_events may not be available on all LlamaIndex versions;
                        # if it fails, we still await the final result below
                        pass
                    final = await handler
                    return steps, str(final)

                steps, final_answer = loop.run_until_complete(collect())

                # Stream each reasoning step to the UI
                for step in steps:
                    accumulated += f"```\n{step}\n```\n\n"
                    yield accumulated

                # Append the final answer
                accumulated += f"---\n\n**Answer:**\n\n{final_answer}"
                yield accumulated

            except Exception as e:
                yield f"❌ Agent error: {type(e).__name__}: {e}"

        # Submit on Enter key press
        agent_input.submit(agent_query_stream, inputs=agent_input, outputs=agent_output)

    demo.launch(share=True)


/bin/bash: line 1: nvidia-smi: command not found
NOT CONNECTED TO A T4


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/tmp/ipykernel_1234/2898632620.py:779: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Analyst-Safe Log Explorer", css=CUSTOM_CSS) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d16f5ce7cf349a2a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Sample questions you can ask the Agent
## Overview

Give me a summary of the dataset
Show me the top 5 errors
Show me the top 10 errors across all services
What is the overall error summary?

## By service

What errors occurred in payment-service?
Show me errors for integration-gateway
Which service has the most errors?
Compare errors between order-service and billing-service

## By error type

Which service had the most NullPointerExceptions?
How many CircuitBreakerOpenExceptions occurred?
Which services are getting SQLExceptions?

## By date

Show errors between 2025-06-01 and 2025-06-30
How many errors occurred in Q1 2025?
What errors happened in January 2026?

## Explanations

Explain what a CircuitBreakerOpenException means
What should I do about OAuthTokenExceptions?
Why would we get a FeignException?

##
Combined

What are the top errors in customer-service and what do they mean?
Show me errors in payment-service from 2025-03-01 to 2025-03-31